[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/02_statistics/02_statistical_rigor.ipynb)

# 02 · 统计严谨性 —— 误差棒、配对检验与功效（动手实验）

<span style="background:#444;color:#fff;border-radius:4px;padding:1px 8px;font-size:12px">CPU</span> 纯 numpy / scipy 模拟，无需模型、无需 GPU、无需 API key。配套讲解：`02_讲解.html`。

本 notebook 把 [Miller 2024, *Adding Error Bars to Evals*, arXiv:2411.00640] 的五条建议全部**亲手实现一遍**：

| 实验 | 你将看到 |
|---|---|
| ① Wald vs Wilson + 覆盖率模拟 | 名义 95% 的区间实际只覆盖 65% 是什么样子 |
| ② percentile bootstrap | 与解析式对照，理解它何时有用 |
| ③ 配对实验（共享难度潜变量） | 同一份数据，未配对检验漏检、配对检验检出 |
| ④ 功效曲线（n = 100 → 5000） | 检出 Δ=3pp 到底要多少题 |
| ⑤ cluster bootstrap | 题目成簇时朴素 SE 低估约 4 倍 |
| ✏️ 练习 ×4 | Wilson 区间 / paired bootstrap p 值 / 样本量公式 / Bonferroni |

**实验设定（贯穿全篇）**：两个"模型"，真实准确率 $p_A=0.70$、$p_B=0.73$（真实分差 $\Delta=3$pp，上帝视角已知；统计推断的任务是只看数据能否把它找回来）。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# ===== 模拟框架：两个"模型" =====
P_A, P_B = 0.70, 0.73   # 真实能力（题目宇宙上的期望准确率）—— 现实中不可知
N = 1000                # benchmark 题目数

# 先看最朴素的设定：两个模型各自独立抽 N 道题作答
# （第 3 节会换成"同一批题"的配对版本——那才是评测的真实场景）
results_a = rng.random(N) < P_A
results_b = rng.random(N) < P_B

acc_a, acc_b = results_a.mean(), results_b.mean()
se_a = np.sqrt(acc_a * (1 - acc_a) / N)
se_b = np.sqrt(acc_b * (1 - acc_b) / N)
print(f"模型 A: acc = {acc_a:.3f} ± {1.96*se_a:.3f}  (95% CI 半宽)")
print(f"模型 B: acc = {acc_b:.3f} ± {1.96*se_b:.3f}")
print(f"观测分差 = {acc_b - acc_a:+.3f}    (真实分差 = {P_B - P_A:+.3f})")

## 1 · Wald 区间与"覆盖率审判"

置信区间是一个**程序**而非一个区间。评判程序好坏的唯一标准是<b>覆盖率</b>（coverage）：
重复整个实验很多次，名义 95% 的区间应当有 ~95% 的次数把真值包进去。

$$\text{Wald: } \hat p \pm z\sqrt{\hat p(1-\hat p)/n}$$

下面的模拟从 $\mathrm{Binomial}(n, p)$ 重复抽样 20000 次，统计 Wald 区间的实际覆盖率。
预期：$n$ 大、$p$ 居中时 ≈0.95；**$n=20,\ p=0.95$ 时塌到 ≈0.65**（区间越界、$\hat p=1$ 时宽度为零等病症的总和）。

In [ ]:
def wald_ci(k, n, z=1.96):
    # Wald 区间。k 可为数组（向量化），便于覆盖率模拟
    p = k / n
    se = np.sqrt(p * (1 - p) / n)
    return p - z * se, p + z * se

def coverage(ci_fn, p_true, n, reps=20000, seed=0):
    # 覆盖率模拟：重复抽样 reps 次，统计名义 95% 区间包住真值的频率
    r = np.random.default_rng(seed)
    ks = r.binomial(n, p_true, size=reps)
    lo, hi = ci_fn(ks, n)
    return np.mean((lo <= p_true) & (p_true <= hi))

print("名义 95% 的 Wald 区间，实际覆盖率：")
for n_, p_ in [(20, 0.95), (50, 0.90), (100, 0.70), (1000, 0.70)]:
    print(f"  n={n_:5d}, p={p_:.2f}:   {coverage(wald_ci, p_, n_):.3f}")

print()
print("→ n 小且 p 极端时（n=20, p=0.95）覆盖率塌到 ~0.65：")
print("  名义 '95% 置信' 是假的。练习 1 的 Wilson 区间会把它修回 ~0.93。")

## 2 · Percentile bootstrap

把经验分布当总体，有放回重采样 $B$ 次，统计量的 $[\alpha/2,\ 1-\alpha/2]$ 分位数即为 percentile CI。

对**均值型**统计量、$n=1000$ 这种规模，bootstrap 与 Wald 几乎重合（下面会验证）——
它的真正价值在于：① 无解析 SE 的复杂统计量（过滤后的 win-rate、宏平均、Elo）；
② 按"独立单位"重采样可以自然推广成 **paired bootstrap**（第 3 节）和 **cluster bootstrap**（第 5 节）。

In [ ]:
def percentile_bootstrap_ci(x, B=10000, alpha=0.05, seed=0):
    # percentile bootstrap：有放回重采样 B 次，取统计量分布的分位数
    r = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    n = len(x)
    idx = r.integers(0, n, size=(B, n))     # 每行是一份重采样的索引
    boot_means = x[idx].mean(axis=1)
    lo, hi = np.quantile(boot_means, [alpha / 2, 1 - alpha / 2])
    return lo, hi

k_a = int(results_a.sum())
w_lo, w_hi = wald_ci(k_a, N)
b_lo, b_hi = percentile_bootstrap_ci(results_a)
print(f"模型 A 的 95% CI（n={N}）：")
print(f"  Wald      : [{w_lo:.4f}, {w_hi:.4f}]")
print(f"  bootstrap : [{b_lo:.4f}, {b_hi:.4f}]")
print("→ 均值 + 大 n：两者几乎重合。bootstrap 的价值在复杂统计量与保持联合结构。")

## 3 · 配对实验：同题成对 vs 各测各的

真实评测中 A、B 跑的是**同一批题**。题目有难度差异，两模型的对错因此**正相关**
（难题一起错、简单题一起对）。配对分析利用这一相关把比较的方差减掉一大块：

$$\mathrm{Var}_{\text{paired}}(\hat\Delta) = \frac{\sigma_A^2+\sigma_B^2- 2\,\mathrm{Cov}(a_i,b_i)}{n} \;<\; \mathrm{Var}_{\text{unpaired}}(\hat\Delta)$$

**数据构造（共享难度潜变量）**：每题一个难度 $z_i\sim\mathcal N(0,1)$，模型 $m$ 的潜变量
$\ell^m_i=\sqrt{\rho}\,z_i+\sqrt{1-\rho}\,\varepsilon^m_i$，答对当且仅当 $\ell^m_i<\Phi^{-1}(p_m)$。
这样边际准确率**精确**等于 $p_m$，而 $\rho$ 控制"共享难度"的强度（取 0.8，对应对错相关 ≈0.55，同档模型常见量级）。

对同一份数据跑四种分析。预期（固定种子 11）：观测分差 ≈ +2.8pp，
**未配对 z 检验 p≈0.17（漏检）**，而配对 z ≈0.04、精确 McNemar ≈0.047、paired bootstrap CI 不含 0 —— **同一份数据，结论相反**。

In [ ]:
def make_paired(n, p_a, p_b, rho, rng):
    # Gaussian copula 构造：共享题目难度 z_i，边际准确率精确等于 p_a / p_b
    z = rng.normal(size=n)
    ea = rng.normal(size=n)
    eb = rng.normal(size=n)
    la = np.sqrt(rho) * z + np.sqrt(1 - rho) * ea
    lb = np.sqrt(rho) * z + np.sqrt(1 - rho) * eb
    return la < stats.norm.ppf(p_a), lb < stats.norm.ppf(p_b)

def unpaired_z_pvalue(a, b):
    # 独立两样本比例 z 检验（pooled SE）—— 故意忽略配对结构的"错误"分析
    n = len(a)
    pa, pb = a.mean(), b.mean()
    pool = (a.sum() + b.sum()) / (2 * n)
    se = np.sqrt(2 * pool * (1 - pool) / n)
    return 2 * stats.norm.sf(abs((pb - pa) / se))

def paired_z_pvalue(a, b):
    # 配对分析：对每题差值 d_i = b_i - a_i 做一样本 z 检验
    d = b.astype(float) - a.astype(float)
    se = d.std(ddof=1) / np.sqrt(len(d))
    return 2 * stats.norm.sf(abs(d.mean() / se))

def mcnemar_pvalue(a, b):
    # 精确 McNemar：只看分歧题，n01 ~ Binomial(n01+n10, 1/2)
    n01 = int(np.sum(~a & b))   # A 错 B 对
    n10 = int(np.sum(a & ~b))   # A 对 B 错
    return stats.binomtest(n01, n01 + n10, 0.5).pvalue, n01, n10

rng_pair = np.random.default_rng(11)
a_p, b_p = make_paired(N, P_A, P_B, rho=0.8, rng=rng_pair)
d = b_p.astype(float) - a_p.astype(float)

p_mc, n01, n10 = mcnemar_pvalue(a_p, b_p)
boot_lo, boot_hi = percentile_bootstrap_ci(d)   # paired bootstrap：按"题"重采样差值

print(f"acc_A = {a_p.mean():.3f},  acc_B = {b_p.mean():.3f},  观测分差 = {d.mean():+.3f}")
print(f"对错相关 corr(a,b) = {np.corrcoef(a_p, b_p)[0, 1]:.2f}    分歧: n01={n01}, n10={n10}")
print()
print(f"未配对 z 检验        p = {unpaired_z_pvalue(a_p, b_p):.3f}   ← 把真实差异淹没在噪声里")
print(f"配对   z 检验        p = {paired_z_pvalue(a_p, b_p):.4f}")
print(f"McNemar（精确）      p = {p_mc:.4f}")
print(f"paired bootstrap 95% CI(Δ) = [{boot_lo:+.3f}, {boot_hi:+.3f}]   (不含 0 ⇒ 显著)")

## 4 · 检出率与功效曲线

单次对比可能碰巧。系统的问法是<b>功效</b>（power）：真实差距 $\Delta=3$pp 时，
检验在 5% 水平上**检出**它的概率是多少？下面对每个 $n$ 模拟 400 个平行宇宙
（每个宇宙重抽一套 $n$ 道题），统计两种检验的检出率。

预期：配对曲线整体在未配对之上——同样 80% 功效，配对约需 $n\approx1700$，未配对约需 $n\approx3500$，
**配对 ≈ 白捡一半以上的题目预算**。也注意 $n\le 500$ 时连配对检验功效也只有 ~0.3：
小 benchmark 上 3pp 的分差，大概率根本测不出来。

In [ ]:
def simulate_power(n, reps, p_a=P_A, p_b=P_B, rho=0.8, seed=7):
    # 向量化模拟：reps 个平行宇宙各抽一套 n 题，返回两种检验的检出率
    r = np.random.default_rng(seed)
    z = r.normal(size=(reps, n))
    la = np.sqrt(rho) * z + np.sqrt(1 - rho) * r.normal(size=(reps, n))
    lb = np.sqrt(rho) * z + np.sqrt(1 - rho) * r.normal(size=(reps, n))
    a = (la < stats.norm.ppf(p_a)).astype(float)
    b = (lb < stats.norm.ppf(p_b)).astype(float)
    # 未配对两比例 z 检验
    pa_h, pb_h = a.mean(axis=1), b.mean(axis=1)
    pool = (pa_h + pb_h) / 2
    se_u = np.sqrt(2 * pool * (1 - pool) / n)
    reject_u = np.abs((pb_h - pa_h) / se_u) > 1.96
    # 配对：每题差值的一样本 z 检验
    dd = b - a
    se_p = dd.std(axis=1, ddof=1) / np.sqrt(n)
    reject_p = np.abs(dd.mean(axis=1) / se_p) > 1.96
    return reject_u.mean(), reject_p.mean()

ns = [100, 250, 500, 1000, 2000, 3500, 5000]
power_u, power_p = [], []
for n_ in ns:
    pu, pp_ = simulate_power(n_, reps=400)
    power_u.append(pu)
    power_p.append(pp_)
    print(f"n={n_:5d}    unpaired 检出率 = {pu:.2f}    paired 检出率 = {pp_:.2f}")

plt.figure(figsize=(7, 4.2))
plt.plot(ns, power_u, "o-", label="unpaired z-test")
plt.plot(ns, power_p, "s-", label="paired z-test (per-question diff)")
plt.axhline(0.8, ls="--", c="gray", lw=1, label="power = 0.8")
plt.xlabel("n (questions)")
plt.ylabel("power")
plt.title("Power to detect a 3pp gap (true 0.70 vs 0.73, latent rho=0.8)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 · Cluster bootstrap：题目成簇时朴素 SE 低估多少

很多 benchmark 的题目**不独立**：一篇文档配多道题、一个模板生成一族题。
独立抽样的单位是"簇"（文档/模板），不是"题"。朴素 SE 公式 $\sqrt{\hat p(1-\hat p)/n}$
默认题目独立，簇内正相关时会**系统性低估**不确定度。两个正确做法：

$$\widehat{\mathrm{Var}}_{\text{clust}}(\hat\theta)=\frac{1}{n^2}\sum_{c}\Big(\sum_{i\in c}(s_i-\hat\theta)\Big)^2 \quad\text{[Miller 2024]}$$

以及 **cluster bootstrap**（以簇为单位有放回重采样）。
设定：20 簇 × 每簇 50 题（$n=1000$），簇内潜变量相关 $\rho_c=0.5$。
预期：朴素 SE ≈0.014，而真实 SE ≈0.06 —— **低估约 4 倍**；解析 clustered SE 与 cluster bootstrap 都能基本找回真值。

In [ ]:
def make_clustered(n_clusters, m, p, rho_c, rng):
    # 同簇共享效应 u_c（同文档/同模板），簇内得分相关；边际准确率精确为 p
    u = rng.normal(size=n_clusters)
    eps = rng.normal(size=n_clusters * m)
    lat = np.sqrt(rho_c) * np.repeat(u, m) + np.sqrt(1 - rho_c) * eps
    correct = lat < stats.norm.ppf(p)
    cluster_id = np.repeat(np.arange(n_clusters), m)
    return correct, cluster_id

def naive_se(x):
    p = x.mean()
    return np.sqrt(p * (1 - p) / len(x))

def clustered_se(x, cluster_id):
    # (1/n²) Σ_c (Σ_{i∈c} (x_i - x̄))²  —— 簇内残差先求和再平方，不让正相关互相抵消
    x = np.asarray(x, dtype=float)
    xb, n = x.mean(), len(x)
    total = 0.0
    for c in np.unique(cluster_id):
        total += (x[cluster_id == c] - xb).sum() ** 2
    return np.sqrt(total) / n

def cluster_bootstrap_se(x, cluster_id, B=4000, seed=0):
    # 以"簇"为重采样单位：抽到一簇就带走整簇题目
    r = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    groups = [x[cluster_id == c] for c in np.unique(cluster_id)]
    G = len(groups)
    means = np.empty(B)
    for bi in range(B):
        pick = r.integers(0, G, size=G)
        means[bi] = np.concatenate([groups[j] for j in pick]).mean()
    return means.std()

G_CL, M_CL, RHO_C = 20, 50, 0.5
rng_cl = np.random.default_rng(3)
x_cl, cid = make_clustered(G_CL, M_CL, 0.70, RHO_C, rng_cl)

# "真实" SE：上帝视角——重新生成整个 benchmark 2000 次，看均值的真实波动
rng_true = np.random.default_rng(9)
true_sd = np.std([make_clustered(G_CL, M_CL, 0.70, RHO_C, rng_true)[0].mean()
                  for _ in range(2000)])

se_n = naive_se(x_cl)
se_c = clustered_se(x_cl, cid)
se_b = cluster_bootstrap_se(x_cl, cid)
print(f"acc = {x_cl.mean():.3f}    ({G_CL} 簇 × 每簇 {M_CL} 题, 簇内相关 rho_c={RHO_C})")
print(f"朴素 SE                 = {se_n:.4f}   ← 假装 1000 题独立")
print(f"clustered SE（解析）    = {se_c:.4f}")
print(f"cluster bootstrap SE    = {se_b:.4f}")
print(f"真实 SE（2000 次重生成）= {true_sd:.4f}")
print(f"→ 朴素 SE 低估约 {true_sd / se_n:.1f} 倍；名义 1000 题的有效样本量只相当于几十道独立题")

## ✏️ 练习 1：实现 `wilson_ci(k, n, z=1.96)`

把第 1 节"覆盖率塌陷"修好。Wilson 区间来自反转 score test（对 $p$ 解二次方程）：

$$\text{denom}=1+\frac{z^2}{n},\qquad \text{center}=\frac{\hat p+\frac{z^2}{2n}}{\text{denom}},\qquad \text{half}=\frac{z}{\text{denom}}\sqrt{\frac{\hat p(1-\hat p)}{n}+\frac{z^2}{4n^2}}$$

返回 `(center - half, center + half)`。

**要求与提示**（约 6 行）：
- 用 numpy 写，保证**向量化**——`k` 可能是 ndarray（自测会把它接进第 1 节的 `coverage` 做覆盖率检验）；
- 对照性质：$k=0$ 时下界恰为 0、$k=n$ 时上界恰为 1（Wald 做不到）。

In [ ]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    # TODO: 按上面的公式实现 denom / center / half，返回 (center - half, center + half)
    raise NotImplementedError("把公式翻译成 numpy（约 4 行）")

In [ ]:
# ---- 练习 1 自测 ----
lo, hi = wilson_ci(8, 10)
assert abs(lo - 0.4902) < 2e-3 and abs(hi - 0.9433) < 2e-3, \
    f"8/10 的 Wilson 区间应为 [0.490, 0.943]，得到 [{lo:.4f}, {hi:.4f}]"

lo0, hi0 = wilson_ci(0, 20)
assert abs(lo0) < 1e-9, "k=0 时下界应恰为 0"
assert 0 < hi0 < 0.25, f"k=0, n=20 的上界应约 0.161，得到 {hi0:.4f}"

lo1, hi1 = wilson_ci(20, 20)
assert abs(hi1 - 1) < 1e-9 and 0.75 < lo1 < 1.0, "k=n 时上界应恰为 1"

w_small = wilson_ci(8, 10)[1] - wilson_ci(8, 10)[0]
w_big = wilson_ci(80, 100)[1] - wilson_ci(80, 100)[0]
assert w_big < w_small, "n 增大 10 倍，区间应明显变窄"

cov_wilson = coverage(wilson_ci, 0.95, 20)   # 覆盖率审判（复用第 1 节的 coverage）
cov_wald = coverage(wald_ci, 0.95, 20)
assert cov_wilson > 0.90, f"Wilson 在 n=20, p=0.95 的覆盖率应 >0.90，得到 {cov_wilson:.3f}"
assert cov_wald < 0.75   # 对照组：Wald 在此场景塌陷
print(f"✅ 练习 1 通过    覆盖率: Wilson {cov_wilson:.3f}  vs  Wald {cov_wald:.3f}")

## ✏️ 练习 2：实现 `paired_bootstrap_pvalue(a_correct, b_correct, B=2000, seed=0)`

把第 3 节的 paired bootstrap 从"看 CI"升级成可直接报告的**双侧 p 值**。算法（请严格按此实现，保证可复现）：

1. `d = b - a`（转 float），`n = len(d)`；
2. `rng = np.random.default_rng(seed)`；一次性生成索引 `idx = rng.integers(0, n, size=(B, n))`，`boot = d[idx].mean(axis=1)`；
3. 双侧 p 值：`p = 2 * min( mean(boot <= 0), mean(boot >= 0) )`，返回 `min(p, 1.0)`。

直觉：bootstrap 分布近似 $\hat\Delta$ 的抽样分布；它压在 0 哪一侧的尾巴越小，差异越显著。约 6 行。

In [ ]:
def paired_bootstrap_pvalue(a_correct, b_correct, B=2000, seed=0):
    d = b_correct.astype(float) - a_correct.astype(float)
    # TODO: 按题重采样 (B, n) 份，计算 boot 均值分布，返回双侧 p 值（见上方算法 2–3 步）
    raise NotImplementedError

In [ ]:
# ---- 练习 2 自测（确定性构造数据）----
a_alt = np.zeros(400, bool); a_alt[:240] = True                    # A: acc 60%
b_alt = a_alt.copy(); b_alt[240:280] = True; b_alt[:5] = False     # B: acc 68.75%（分歧 40↑ vs 5↓）
a_nul = np.zeros(400, bool); a_nul[:240] = True
b_nul = a_nul.copy(); b_nul[240:260] = True; b_nul[:20] = False    # 同 acc（分歧 20↑ vs 20↓）

p_alt = paired_bootstrap_pvalue(a_alt, b_alt, B=2000)
p_nul = paired_bootstrap_pvalue(a_nul, b_nul, B=2000)
assert 0 <= p_alt <= 1 and 0 <= p_nul <= 1
assert p_alt < 0.01, f"B 明显更强（+8.75pp，分歧 40:5），应高度显著，得到 p={p_alt}"
assert p_nul > 0.2, f"两模型打平（分歧 20:20），不应显著，得到 p={p_nul}"
assert paired_bootstrap_pvalue(b_alt, a_alt, B=2000) == p_alt, "双侧 p 值应对 A/B 交换对称"
print(f"✅ 练习 2 通过    p_alt = {p_alt:.4f},  p_null = {p_nul:.4f}")

## ✏️ 练习 3：实现 `n_required(delta, p_base, power=0.8, alpha=0.05)`

功效公式写成函数：**未配对**双侧检验下，每个模型需要多少题才能以给定功效检出 $\Delta$：

$$n=\left\lceil \frac{2\,\bar p(1-\bar p)\,\big(z_{1-\alpha/2}+z_{\text{power}}\big)^2}{\Delta^2} \right\rceil,\qquad \bar p=p_{\text{base}}+\frac{\Delta}{2}$$

**提示**（约 5 行）：分位数用 `stats.norm.ppf(1 - alpha/2)` 和 `stats.norm.ppf(power)`；
向上取整 `np.ceil` 后转 `int` 返回。

In [ ]:
def n_required(delta, p_base, power=0.8, alpha=0.05):
    # TODO: 实现上面的公式，返回 int（每个模型所需题数）
    raise NotImplementedError

In [ ]:
# ---- 练习 3 自测 ----
n_2pp = n_required(0.02, 0.70)
n_3pp = n_required(0.03, 0.70)
n_5pp = n_required(0.05, 0.70)
assert isinstance(n_2pp, (int, np.integer)), "应返回整数题数"
assert abs(n_2pp - 8081) <= 3, f"Δ=2pp, p=0.70, power=0.8 → 应约 8081/模型，得到 {n_2pp}"
assert abs(n_3pp - 3555) <= 3, f"Δ=3pp → 应约 3555，得到 {n_3pp}"
assert n_2pp > n_3pp > n_5pp, "Δ 越小所需 n 越大（n ∝ 1/Δ²）"
assert n_required(0.03, 0.70, power=0.9) > n_3pp, "功效要求越高，所需 n 越大"
assert abs(n_2pp / n_required(0.04, 0.70) - 4) < 0.15, "Δ 减半 → n 应约 4 倍"
print(f"✅ 练习 3 通过    n(2pp)={n_2pp}, n(3pp)={n_3pp}, n(5pp)={n_5pp}")
print("→ 回看讲解第 2 节的表：GPQA-diamond 只有 198 题，对 2–3pp 的差距毫无功效。")

## ✏️ 练习 4（简单）：Bonferroni 校正

跑 $m$ 个 benchmark 做 $m$ 次检验，FWER $=1-(1-\alpha)^m$（$m=20$ 时 ≈64%）。
实现 `bonferroni(pvals, alpha=0.05)`，返回元组 `(reject, p_adj)`：

- `p_adj = min(p * m, 1.0)`（逐元素，校正后 p 值截断在 1）；
- `reject = p < alpha / m`（布尔数组，注意用**原始** p 与 α/m 比较）。

约 4 行：`np.asarray` / `np.minimum` 即可。

In [ ]:
def bonferroni(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float)
    # TODO: 计算 reject（bool 数组）与 p_adj（截断在 1），返回 (reject, p_adj)
    raise NotImplementedError

In [ ]:
# ---- 练习 4 自测 ----
pv = np.array([0.001, 0.01, 0.03, 0.2])
rej, adj = bonferroni(pv, alpha=0.05)
assert list(rej) == [True, True, False, False], \
    f"m=4 时阈值为 0.05/4=0.0125，应为 [T,T,F,F]，得到 {list(rej)}"
assert np.allclose(adj, [0.004, 0.04, 0.12, 0.8])

rej2, adj2 = bonferroni([0.5, 0.9])
assert np.allclose(adj2, [1.0, 1.0]), "校正后 p 值应截断在 1"
assert not np.any(rej2)

rej3, adj3 = bonferroni([0.04])
assert rej3[0] and np.allclose(adj3, [0.04]), "m=1 时应退化为普通检验"
print("✅ 练习 4 通过")

## 📖 参考答案

先自己做，再对照。每题都只有几行——如果你的实现明显更长，回头想想哪里绕了远路。

In [ ]:
# 参考答案 1 —— 先自己做，再对照
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z / denom * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return center - half, center + half

In [ ]:
# 参考答案 2 —— 先自己做，再对照
def paired_bootstrap_pvalue(a_correct, b_correct, B=2000, seed=0):
    d = b_correct.astype(float) - a_correct.astype(float)
    n = len(d)
    r = np.random.default_rng(seed)
    idx = r.integers(0, n, size=(B, n))
    boot = d[idx].mean(axis=1)
    p = 2 * min(np.mean(boot <= 0), np.mean(boot >= 0))
    return min(p, 1.0)

In [ ]:
# 参考答案 3 —— 先自己做，再对照
def n_required(delta, p_base, power=0.8, alpha=0.05):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_power = stats.norm.ppf(power)
    p_bar = p_base + delta / 2
    n = 2 * p_bar * (1 - p_bar) * (z_alpha + z_power) ** 2 / delta**2
    return int(np.ceil(n))

In [ ]:
# 参考答案 4 —— 先自己做，再对照
def bonferroni(pvals, alpha=0.05):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    p_adj = np.minimum(p * m, 1.0)
    reject = p < alpha / m
    return reject, p_adj

## 小结

把本 notebook 浓缩成六条肌肉记忆（对应 [Miller 2024] 的报告规范）：

1. **任何 eval 分数都报 95% CI**；$n\lesssim100$ 或 $\hat p$ 极端时不要用 Wald，用 Wilson（练习 1：覆盖率 0.65 → 0.93）。
2. **复杂统计量用 percentile bootstrap**；它是通用性工具，不是精度工具——救不了 $n$ 太小。
3. **模型对比永远分析每题差值** $d_i$：paired SE / McNemar / paired bootstrap（练习 2）。同一份数据，未配对 p=0.17、配对 p=0.04——别让自己的真实提升被自己的错误检验淹没。
4. **题目成簇时按簇算 SE**：朴素 SE 可低估 4 倍；重采样单位 = 独立性单位。
5. **跑 eval 之前先算功效**（练习 3）：检出 2pp 差距需约 8000 题（未配对）；小 benchmark 上的小分差 = 没有测量。
6. **跑了 20 个 benchmark 就要做多重校正**（练习 4）：高风险结论用 Bonferroni，大规模筛查用 BH/FDR。

**但是**——本章所有 CI 只覆盖了"题目抽样"这一种随机性。换一个 prompt 模板，分数可以**系统性**平移十几个百分点，
再多的题、再宽的误差棒都不反映它。这就是下一模块的主题：

→ **模块 03 · 答案抽取与 prompt 敏感性**（`../03_prompt_sensitivity/03_讲解.html`）

---
## 🎯 真实数据胶囊题：真实 benchmark 上两个模型差异的配对 bootstrap CI

比较两个模型，必须用**配对**设计（同题）并报置信区间。用真实 GSM8K 题目构造两个模型的逐题正确性，实现配对 bootstrap：重采样题目、算差值分布、给 95% CI，判断差异是否显著。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

rows=gsm8k(200); n=len(rows)
rng=np.random.default_rng(0)
# 两个模型在真实题目上的逐题正确性（B 略强）
A=(rng.random(n)<0.55).astype(float)
B=A.copy(); flip=rng.random(n)<0.12; B[flip]=1-B[flip]   # B 在部分题上不同
B=np.clip(B + (rng.random(n)<0.05), 0, 1)
print(f"{n} 道真实题  A acc={A.mean():.3f}  B acc={B.mean():.3f}")

**练习**：实现 `paired_bootstrap_ci(A, B, n_boot)`：对**题目索引**有放回重采样，每次算 `mean(B)-mean(A)`，返回 `(点估计, lo, hi)` 的 95% CI。

In [ ]:
def paired_bootstrap_ci(A, B, n_boot=2000, seed=0):
    # TODO: 重采样题目索引(配对!)，算差值分布，取2.5/97.5分位
    raise NotImplementedError


In [ ]:
# 自测
pt,lo,hi = paired_bootstrap_ci(A,B,2000)
assert abs(pt-(B.mean()-A.mean()))<1e-9
assert lo<=pt<=hi and hi>lo
# 自己跟自己比，差异为0且CI含0
p2,l2,h2=paired_bootstrap_ci(A,A,2000)
assert abs(p2)<1e-9 and l2<=0<=h2
print(f"B-A 差异 = {pt:+.3f}  95% CI=[{lo:+.3f},{hi:+.3f}]  显著={'是' if lo>0 else '否(CI含0)'}")


### 📖 参考答案

In [ ]:
def paired_bootstrap_ci(A, B, n_boot=2000, seed=0):
    A=np.asarray(A,float); B=np.asarray(B,float); n=len(A); rng=np.random.default_rng(seed)
    diffs=np.array([ (B[idx].mean()-A[idx].mean()) for idx in (rng.integers(0,n,n) for _ in range(n_boot))])
    lo,hi=np.percentile(diffs,[2.5,97.5])
    return float(B.mean()-A.mean()), float(lo), float(hi)
print("✓ 配对 + bootstrap：模型对比的标准口径")